from 2022

# 工作流程

## 基本理论

通过一种特殊的提示工程来引导模型

每一步的输出分为
- reason
- action
- observation

不断重复 reason -> action -> observation , 直到触发预定义的终止条件(thought中内容符合要求), 公式化的表达如下: 

$$
(t h_t, a_t) = \pi \bigl(q, (a_1, o_1), \dots, (a_{t-1}, o_{t-1})\bigr)
$$

$$
o_t = T(a_t)
$$

**适用场景**
- 需要补充外部信息来完成决策(知识, 静态)
- 需要与外部完成信息交互(交互, 动态)
- 需要完成精确的计算任务(工具调用)

## action的方式-tool
三要素
- 名称
- 描述: 让llm理解这个工具
- 实体: 可以执行的函数或者方法

## 特点
- 武装llm, 格外重视工具调用
- 可解释性强
- 步步为营, 稳扎稳打

## 局限性
- 缺乏全局规划: 很容易陷入局部最优解或者陷入局部循环
- 高度依赖底层模型和prompt
- 串行调用, 执行效率低

In [1]:
%cd ../
import os
print(os.getcwd())

/Users/yqz/project/hello-agents
/Users/yqz/project/hello-agents


### 协议设计

llm_output格式: 参考上一章节的; 
```txt
{
  "thought": "...",
  "todo": {
    "name": "<what to do>",
    "args": { <param_name>: <param_value> }
  }
}
```
比如
终止条件为
```txt
{
  "thought": "...",
  "todo": {
    "name": "finish",
    "args": { "<param_name>": "<param_value>"}
  }
}
```
某个函数调用为, 它的description为"search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information."
```txt
{
  "thought": "...",
  "todo": {
    "name": "search",
    "args": { "query": "Who is the current President of the United States?"  }
  }
}
```

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# ReActAgent类
import json
class ReActAgent:
    def __init__(self, llm, tool, max_step):
        self.client = llm
        self.tool = tool
        self.max_step = max_step
        
        self.history = []

    def run(self, prompt, system_prompt):
        self.history = []
        current_step = 0
        self.history.append(
            json.dumps({"role": "user", "content": prompt}, ensure_ascii=False)
        )
        while current_step < self.max_step:
            print('='*20 + f"第{current_step+1}轮" + '='*20)
            
            # print(f'input: \n{self.history}\n')
            full_prompt = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": "\n".join(self.history)}
            ]
            output = self.client.generate(
                messages=full_prompt
            )
            # print(f'output: \n{output}\n')

            result = json.loads(output)
            '''
            复习一下协议
            {
                "thought": "...",
                "todo": {
                    "name": "<what to do>",
                    "args": { <param_name>: <param_value> }
                }
            }
            '''
            if result.get("todo").get("name") == "finish":
                print(f'\n {result["todo"]["args"]} \n')
                print("finished")
                break
            else:
                args = result.get("todo").get("args")
                tool_function = self.tool.useTool(
                    result.get("todo").get("name")
                )
                tool_result = tool_function(**args)
                print(f'tool_result: \n{tool_result}\n')
                self.history.append(
                    json.dumps({"role": "tool", "content": tool_result}, ensure_ascii=False)
                )
            current_step += 1
        if current_step >= self.max_step:
            print("reach max step")

### 提示词构建
提供两个可供替代的内容 `question` 和 `introduceTool`

PS: 刚刚遇到个问题
json自带的 {}, 影响了 string.format的使用, 这里我改用 Template 模块解决

In [4]:
from string import Template
REACT_PROMPT = Template("""
You are a Tool-Calling Agent. Decide the single next action to solve the user question using ONLY the tools listed below, or finish with a final answer.

INPUT
- User question:
$question

- Available tools (you may ONLY use tools listed here; includes name, args, and description):
$introduceTool

OUTPUT FORMAT (must follow EXACTLY)
Return ONLY one valid JSON object:

{
  "thought": "...",
  "todo": {
    "name": "<tool_name_or_finish>",
    "args": { ... }
  }
}

RULES
1) Output ONLY JSON. No markdown, no extra text.
2) "thought" must be a short, high-level plan (1–2 sentences). Do NOT include detailed step-by-step reasoning.
3) If you choose a tool:
   - todo.name MUST be exactly one tool name from {introuceTool}.
   - todo.args MUST match that tool’s required parameters (correct names and types). Do not add extra params.
4) If you finish:
   - todo.name MUST be "finish"
   - todo.args MUST be: { "answer": "<final response to the user question>" }
5) Do NOT invent facts or tool results. If the question needs up-to-date or external info, call the most relevant tool first.
"""
)
REACT_PROMPT

In [5]:
# 应用提示词模版的示例
print(REACT_PROMPT.substitute(question="111", introduceTool="222"))


You are a Tool-Calling Agent. Decide the single next action to solve the user question using ONLY the tools listed below, or finish with a final answer.

INPUT
- User question:
111

- Available tools (you may ONLY use tools listed here; includes name, args, and description):
222

OUTPUT FORMAT (must follow EXACTLY)
Return ONLY one valid JSON object:

{
  "thought": "...",
  "todo": {
    "name": "<tool_name_or_finish>",
    "args": { ... }
  }
}

RULES
1) Output ONLY JSON. No markdown, no extra text.
2) "thought" must be a short, high-level plan (1–2 sentences). Do NOT include detailed step-by-step reasoning.
3) If you choose a tool:
   - todo.name MUST be exactly one tool name from {introuceTool}.
   - todo.args MUST match that tool’s required parameters (correct names and types). Do not add extra params.
4) If you finish:
   - todo.name MUST be "finish"
   - todo.args MUST be: { "answer": "<final response to the user question>" }
5) Do NOT invent facts or tool results. If the questi

### 整理工具

In [6]:
from mylib import ToolExecutor, SEARCH_DESCRIPTION, search

# 初始化工具管理器
my_tool = ToolExecutor()

# 新建search工具(一个函数)
mysearch = search

# 加入到工具管理器
my_tool.registerTool(
    name="search",
    description=SEARCH_DESCRIPTION,
    function=mysearch
)

# 生成工具管理器初始化描述
introduceTool = my_tool.introduceTool()
print(introduceTool)

- finish(conclusion: str): Call this action when you think it's time to end, args is your conclusion
- search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information.


### 开始测试

In [7]:
# 整合输入
input = "tell me the latest price of gold"

In [8]:
system_prompt = REACT_PROMPT.substitute(
    introduceTool = introduceTool,
    question = input
)
print(system_prompt)


You are a Tool-Calling Agent. Decide the single next action to solve the user question using ONLY the tools listed below, or finish with a final answer.

INPUT
- User question:
tell me the latest price of gold

- Available tools (you may ONLY use tools listed here; includes name, args, and description):
- finish(conclusion: str): Call this action when you think it's time to end, args is your conclusion
- search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information.

OUTPUT FORMAT (must follow EXACTLY)
Return ONLY one valid JSON object:

{
  "thought": "...",
  "todo": {
    "name": "<tool_name_or_finish>",
    "args": { ... }
  }
}

RULES
1) Output ONLY JSON. No markdown, no extra text.
2) "thought" must be a short, high-level plan (1–2 sentences). Do NOT include detailed step-by-step reasoning.
3) If you choose a tool:
   - todo.name MUST be exactly one tool name from {int

In [9]:
from mylib import MyLLM
my_llm = MyLLM()
agent = ReActAgent(
    llm=my_llm,
    tool=my_tool,
    max_step=10
)
agent.run(
    prompt=input,
    system_prompt=system_prompt
)

====================第1轮====================
llm输入: 
[{'role': 'system', 'content': '\nYou are a Tool-Calling Agent. Decide the single next action to solve the user question using ONLY the tools listed below, or finish with a final answer.\n\nINPUT\n- User question:\ntell me the latest price of gold\n\n- Available tools (you may ONLY use tools listed here; includes name, args, and description):\n- finish(conclusion: str): Call this action when you think it\'s time to end, args is your conclusion\n- search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information.\n\nOUTPUT FORMAT (must follow EXACTLY)\nReturn ONLY one valid JSON object:\n\n{\n  "thought": "...",\n  "todo": {\n    "name": "<tool_name_or_finish>",\n    "args": { ... }\n  }\n}\n\nRULES\n1) Output ONLY JSON. No markdown, no extra text.\n2) "thought" must be a short, high-level plan (1–2 sentences). Do NOT include det

### 记录一次llm的反思能力
```bash
====================第0轮====================
llm输入: 
[{'role': 'system', 'content': '\nYou are a Tool-Calling Agent. Decide the single next action to solve the user question using ONLY the tools listed below, or finish with a final answer.\n\nINPUT\n- User question:\ntell me the latest price of gold\n\n- Available tools (you may ONLY use tools listed here; includes name, args, and description):\n- finish(conclusion: str): Call this action when you think it\'s time to end, args is your conclusion\n- search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information.\n\nOUTPUT FORMAT (must follow EXACTLY)\nReturn ONLY one valid JSON object:\n\n{\n  "thought": "...",\n  "todo": {\n    "name": "<tool_name_or_finish>",\n    "args": { ... }\n  }\n}\n\nRULES\n1) Output ONLY JSON. No markdown, no extra text.\n2) "thought" must be a short, high-level plan (1–2 sentences). Do NOT include detailed step-by-step reasoning.\n3) If you choose a tool:\n   - todo.name MUST be exactly one tool name from {introuceTool}.\n   - todo.args MUST match that tool’s required parameters (correct names and types). Do not add extra params.\n4) If you finish:\n   - todo.name MUST be "finish"\n   - todo.args MUST be: { "answer": "<final response to the user question>" }\n5) Do NOT invent facts or tool results. If the question needs up-to-date or external info, call the most relevant tool first.\n'}, {'role': 'user', 'content': '{"role": "user", "content": "tell me the latest price of gold"}'}]

llm输出: 
{
  "thought": "I need to search for the latest gold price using a web search engine.",
  "todo": {
    "name": "search",
    "args": {
      "query": "latest gold price per ounce"
    }
  }
}

正在执行[SerpApi]网页搜索: latest gold price per ounce
tool_result: 
[1] Gold Price Today - Live Gold Spot Price Charts
Gold is priced at $4,705.72 per ounce, stabilizing near the $4,700 level after last week's steep Warsh‑driven selloff pushed prices down from record highs ...

[2] Gold Price per Ounce
On this page you can view the current price of gold per ounce, gram or kilo. Gold is usually quoted by the ounce in U.S. Dollars. The gold price can, ...

[3] Live Gold Spot Price | Price Charts per Ounce/Gram
Live Gold Spot Prices ; Gold Prices Per Ounce, $4,894.00. -529.00 ; Gold Prices Per Gram, $157.35. -17.01 ; Gold Prices Per Kilo, $157,342.10. -17007.35 ...

====================第1轮====================
llm输入: 
[{'role': 'system', 'content': '\nYou are a Tool-Calling Agent. Decide the single next action to solve the user question using ONLY the tools listed below, or finish with a final answer.\n\nINPUT\n- User question:\ntell me the latest price of gold\n\n- Available tools (you may ONLY use tools listed here; includes name, args, and description):\n- finish(conclusion: str): Call this action when you think it\'s time to end, args is your conclusion\n- search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information.\n\nOUTPUT FORMAT (must follow EXACTLY)\nReturn ONLY one valid JSON object:\n\n{\n  "thought": "...",\n  "todo": {\n    "name": "<tool_name_or_finish>",\n    "args": { ... }\n  }\n}\n\nRULES\n1) Output ONLY JSON. No markdown, no extra text.\n2) "thought" must be a short, high-level plan (1–2 sentences). Do NOT include detailed step-by-step reasoning.\n3) If you choose a tool:\n   - todo.name MUST be exactly one tool name from {introuceTool}.\n   - todo.args MUST match that tool’s required parameters (correct names and types). Do not add extra params.\n4) If you finish:\n   - todo.name MUST be "finish"\n   - todo.args MUST be: { "answer": "<final response to the user question>" }\n5) Do NOT invent facts or tool results. If the question needs up-to-date or external info, call the most relevant tool first.\n'}, {'role': 'user', 'content': '{"role": "user", "content": "tell me the latest price of gold"}\n{"role": "tool", "content": "[1] Gold Price Today - Live Gold Spot Price Charts\\nGold is priced at $4,705.72 per ounce, stabilizing near the $4,700 level after last week\'s steep Warsh‑driven selloff pushed prices down from record highs ...\\n\\n[2] Gold Price per Ounce\\nOn this page you can view the current price of gold per ounce, gram or kilo. Gold is usually quoted by the ounce in U.S. Dollars. The gold price can, ...\\n\\n[3] Live Gold Spot Price | Price Charts per Ounce/Gram\\nLive Gold Spot Prices ; Gold Prices Per Ounce, $4,894.00. -529.00 ; Gold Prices Per Gram, $157.35. -17.01 ; Gold Prices Per Kilo, $157,342.10. -17007.35 ..."}'}]

llm输出: 
{
  "thought": "The search results show conflicting gold prices ($4,705.72 and $4,894.00 per ounce), likely due to different sources or timestamps. I need to clarify the most recent and reliable spot price.",
  "todo": {
    "name": "search",
    "args": {
      "query": "current live gold spot price per ounce USD"
    }
  }
}

正在执行[SerpApi]网页搜索: current live gold spot price per ounce USD
tool_result: 
$4,894.00

====================第2轮====================
llm输入: 
[{'role': 'system', 'content': '\nYou are a Tool-Calling Agent. Decide the single next action to solve the user question using ONLY the tools listed below, or finish with a final answer.\n\nINPUT\n- User question:\ntell me the latest price of gold\n\n- Available tools (you may ONLY use tools listed here; includes name, args, and description):\n- finish(conclusion: str): Call this action when you think it\'s time to end, args is your conclusion\n- search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information.\n\nOUTPUT FORMAT (must follow EXACTLY)\nReturn ONLY one valid JSON object:\n\n{\n  "thought": "...",\n  "todo": {\n    "name": "<tool_name_or_finish>",\n    "args": { ... }\n  }\n}\n\nRULES\n1) Output ONLY JSON. No markdown, no extra text.\n2) "thought" must be a short, high-level plan (1–2 sentences). Do NOT include detailed step-by-step reasoning.\n3) If you choose a tool:\n   - todo.name MUST be exactly one tool name from {introuceTool}.\n   - todo.args MUST match that tool’s required parameters (correct names and types). Do not add extra params.\n4) If you finish:\n   - todo.name MUST be "finish"\n   - todo.args MUST be: { "answer": "<final response to the user question>" }\n5) Do NOT invent facts or tool results. If the question needs up-to-date or external info, call the most relevant tool first.\n'}, {'role': 'user', 'content': '{"role": "user", "content": "tell me the latest price of gold"}\n{"role": "tool", "content": "[1] Gold Price Today - Live Gold Spot Price Charts\\nGold is priced at $4,705.72 per ounce, stabilizing near the $4,700 level after last week\'s steep Warsh‑driven selloff pushed prices down from record highs ...\\n\\n[2] Gold Price per Ounce\\nOn this page you can view the current price of gold per ounce, gram or kilo. Gold is usually quoted by the ounce in U.S. Dollars. The gold price can, ...\\n\\n[3] Live Gold Spot Price | Price Charts per Ounce/Gram\\nLive Gold Spot Prices ; Gold Prices Per Ounce, $4,894.00. -529.00 ; Gold Prices Per Gram, $157.35. -17.01 ; Gold Prices Per Kilo, $157,342.10. -17007.35 ..."}\n{"role": "tool", "content": "$4,894.00"}'}]

llm输出: 
{
  "thought": "The latest gold price is available from the search results, showing $4,894.00 per ounce.",
  "todo": {
    "name": "finish",
    "args": {
      "conclusion": "The latest price of gold is $4,894.00 per ounce."
    }
  }
}


 {'conclusion': 'The latest price of gold is $4,894.00 per ounce.'} 

finished
```